In [1]:
# %%
import pandas as pd

df = pd.read_csv("/home/thinkpad/Documents/BrototypeStudying/Paper two/week 8/code/data/raw_employee_data.csv")

# %%
# Drop rows with no target — can't train/evaluate without salary
df = df.dropna(subset="salary")

# %%
# Drop only truly non-predictive columns — keep performance_rating AND education this time
df = df.drop(columns=["employee_id", "name", "email", "phone", "join_date", "notes"])

# %%
from sklearn.model_selection import train_test_split

x = df.drop(columns=["salary"])
y = df["salary"]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

# %%
# --- Clean age ---
median_age = x_train["age"].median()

x_train.loc[x_train["age"] > 100, "age"] = median_age
x_test.loc[x_test["age"] > 100, "age"] = median_age

x_train.loc[x_train["age"] < 18, "age"] = median_age
x_test.loc[x_test["age"] < 18, "age"] = median_age

x_train["age"] = x_train["age"].fillna(median_age)
x_test["age"] = x_test["age"].fillna(median_age)

# %%
# --- Clean department ---
department_mode = x_train["department"].mode()[0]
x_train["department"] = x_train["department"].fillna(department_mode)
x_test["department"] = x_test["department"].fillna(department_mode)

x_train["department"] = x_train["department"].str.strip().str.lower()
x_test["department"] = x_test["department"].str.strip().str.lower()

dept_map = {
    "suport": "support",
    "i.t.": "it",
    "human resources": "hr",
    "h.r.": "hr",
    "markting": "marketing",
    "ops": "operations"
}
x_train["department"] = x_train["department"].replace(dept_map)
x_test["department"] = x_test["department"].replace(dept_map)

# %%
# --- Clean city ---
city_mode = x_train["city"].mode()[0]
x_train["city"] = x_train["city"].fillna(city_mode)
x_test["city"] = x_test["city"].fillna(city_mode)

x_train["city"] = x_train["city"].str.strip().str.lower()
x_test["city"] = x_test["city"].str.strip().str.lower()

city_map = {"nyc": "new york"}
x_train["city"] = x_train["city"].replace(city_map)
x_test["city"] = x_test["city"].replace(city_map)

# %%
# --- Clean years_experience ---
median_exp = x_train["years_experience"].median()
x_train["years_experience"] = x_train["years_experience"].fillna(median_exp)
x_test["years_experience"] = x_test["years_experience"].fillna(median_exp)

invalid_train = x_train["years_experience"] > (x_train["age"] - 18)
x_train.loc[invalid_train, "years_experience"] = (x_train.loc[invalid_train, "age"] - 18).clip(lower=0)

invalid_test = x_test["years_experience"] > (x_test["age"] - 18)
x_test.loc[invalid_test, "years_experience"] = (x_test.loc[invalid_test, "age"] - 18).clip(lower=0)

# %%
# --- Clean remote_work ---
mode_remote = x_train["remote_work"].mode()[0]
x_train["remote_work"] = x_train["remote_work"].fillna(mode_remote)
x_test["remote_work"] = x_test["remote_work"].fillna(mode_remote)

# %%
# --- Clean performance_rating ---
median_perf = x_train["performance_rating"].median()
x_train["performance_rating"] = x_train["performance_rating"].fillna(median_perf)
x_test["performance_rating"] = x_test["performance_rating"].fillna(median_perf)

# %%
# --- Clean education (NEW) ---
edu_mode = x_train["education"].mode()[0]
x_train["education"] = x_train["education"].fillna(edu_mode)
x_test["education"] = x_test["education"].fillna(edu_mode)

x_train["education"] = x_train["education"].str.strip().str.lower()
x_test["education"] = x_test["education"].str.strip().str.lower()

print(x_train["education"].unique())  # check for typos/variants before encoding

# %%
# --- Clean gender (NEW, optional — check correlation first before deciding to keep) ---
gender_mode = x_train["gender"].mode()[0]
x_train["gender"] = x_train["gender"].fillna(gender_mode)
x_test["gender"] = x_test["gender"].fillna(gender_mode)
x_train["gender"] = x_train["gender"].str.strip().str.lower()
x_test["gender"] = x_test["gender"].str.strip().str.lower()

# %%
# --- One-hot encode all categorical columns ---
cat_cols = ["department", "city", "education", "gender"]

x_train = pd.get_dummies(x_train, columns=cat_cols, drop_first=True).astype(int)
x_test = pd.get_dummies(x_test, columns=cat_cols, drop_first=True).astype(int)

# Align columns in case a category appears only in train or only in test
x_train, x_test = x_train.align(x_test, join="left", axis=1, fill_value=0)

# %%
# --- Clean salary (target) ---
y_train = y_train.astype(str).str.replace(r'[^\d.]', '', regex=True).astype(float)
y_test = y_test.astype(str).str.replace(r'[^\d.]', '', regex=True).astype(float)

median_salary = y_train.median()
y_train[y_train > 200000] = median_salary
y_test[y_test > 200000] = median_salary

# %%
# --- Scale numeric columns ---
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
cols_to_scale = ["age", "years_experience", "performance_rating"]

x_train[cols_to_scale] = scaler.fit_transform(x_train[cols_to_scale])
x_test[cols_to_scale] = scaler.transform(x_test[cols_to_scale])   # transform only, not fit_transform

# %%
# --- Train Linear Regression ---
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

model = LinearRegression()
model.fit(x_train, y_train)
pred = model.predict(x_test)

print("Linear Regression")
print("MAE :", mean_absolute_error(y_test, pred))
print("R2  :", r2_score(y_test, pred))

# %%
# --- Try Random Forest too, compare ---
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(n_estimators=300, max_depth=8, random_state=42)
rf_model.fit(x_train, y_train)
rf_pred = rf_model.predict(x_test)

print("\nRandom Forest")
print("MAE :", mean_absolute_error(y_test, rf_pred))
print("R2  :", r2_score(y_test, rf_pred))

# %%
# --- Feature importance (Random Forest) ---
importances = pd.Series(rf_model.feature_importances_, index=x_train.columns).sort_values(ascending=False)
print("\nTop features:")
print(importances.head(10))

<StringArray>
['phd', 'master', 'bachelor', 'high school', 'bachelors']
Length: 5, dtype: str
Linear Regression
MAE : 4984.053955324428
R2  : 0.7972132796663204

Random Forest
MAE : 6416.622680979887
R2  : 0.7292377324161828

Top features:
years_experience         0.261368
education_phd            0.215835
department_support       0.106550
department_hr            0.060808
department_operations    0.049245
age                      0.043562
department_it            0.036027
department_sales         0.033582
performance_rating       0.030199
department_marketing     0.029010
dtype: float64
